# PSI playground

Песочница для бининга фичей (PSI / WoE) и мониторинга стабильности.

| Файл | Назначение |
|------|------------|
| `synthetic_data.py` | генерация выборки (12k строк, 12 месяцев, 7 фичей) |
| `binning/` | пакет: `NumBinner` / `CatBinner` (fit/transform), независим от PSI |
| `psi.py` | `define_base_data` (выбор базы) + `calc_psi_by_features` / `calc_psi_by_period` |
| `psi_plots.py` | графики plotly (распределения линиями + PSI) |

Запусти ячейки сверху вниз. Kernel — **Python 3.14 (.venv)**.

In [65]:
import sys, pathlib, importlib
sys.path.insert(0, str(pathlib.Path.cwd()))

import numpy as np
import pandas as pd

import synthetic_data, psi, psi_plots
from binning import NumBinner, CatBinner
for m in (synthetic_data, psi, psi_plots):
    importlib.reload(m)

# тип фичи задаётся ЯВНО (не по префиксу/dtype)
NUMERIC = ['f__income', 'f__score', 'f__txn_count', 'f__rate', 'f__num_products']
CATEGORICAL = ['c__region', 'c__channel']
FEATURES = NUMERIC + CATEGORICAL

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
print('modules loaded')

modules loaded


## 1. Данные

In [67]:
from pathlib import Path
DATA = Path('data/sample.parquet')
if not DATA.exists():
    synthetic_data.save_sample(synthetic_data.generate_sample(n_total=12_000, seed=42), str(DATA))
df = synthetic_data.load_sample(str(DATA))
print('shape:', df.shape)
df.head()

shape: (12000, 9)


,sample_date_orig,sample_month,f__income,f__score,f__txn_count,f__rate,f__num_products,c__region,c__channel
0,2024-01-20,2024-01-01,41489.104031,0.983886,3,0.784482,1,South,partner
1,2024-01-22,2024-01-01,89760.171142,0.945670,0,0.500000,2,South,partner
2,2024-01-04,2024-01-01,26681.140861,0.456750,7,0.500000,1,Central,aggregator_16
3,2024-01-10,2024-01-01,50174.628458,0.266843,5,0.500000,1,West,partner
4,2024-01-24,2024-01-01,208667.808624,0.117256,4,0.779693,4,East,branch


In [68]:
df['sample_month'].value_counts().sort_index()

sample_month
2024-01-01     904
2024-02-01     849
2024-03-01    1022
2024-04-01     983
2024-05-01     853
2024-06-01     860
2024-07-01    1034
2024-08-01    1042
2024-09-01    1041
2024-10-01     994
2024-11-01    1189
2024-12-01    1229
Name: count, dtype: int64

## 2. Профиль фичей и краевые случаи
Каждая фича сконструирована под свой краевой случай бининга.

In [69]:
df[synthetic_data.FEATURES_NUMERIC].describe()

,f__income,f__score,f__txn_count,f__rate,f__num_products
count,11526.000000,12000.000000,12000.000000,12000.000000,12000.000000
mean,66884.811799,0.499365,1.836667,0.498924,2.405833
std,40382.448935,0.204443,2.315909,0.165165,1.336893
min,7678.053410,0.011551,0.000000,0.015508,1.000000
25%,39718.353217,0.344882,0.000000,0.427602,1.000000
50%,57628.560807,0.500138,0.000000,0.500000,2.000000
75%,82582.649053,0.654980,4.000000,0.568230,3.000000
max,538213.881839,0.992131,12.000000,0.995981,5.000000


In [70]:
print('f__income    доля NaN   =', round(df['f__income'].isna().mean(), 3))
print('f__txn_count  доля нулей =', round((df['f__txn_count'] == 0).mean(), 3))
print('f__rate       доля 0.5   =', round((df['f__rate'] == 0.5).mean(), 3))
print('f__num_products уникальные =', sorted(df['f__num_products'].unique().tolist()))
print('c__channel    уник. кат. =', df['c__channel'].nunique())

f__income    доля NaN   = 0.04
f__txn_count  доля нулей = 0.539
f__rate       доля 0.5   = 0.347
f__num_products уникальные = [1, 2, 3, 4, 5]
c__channel    уник. кат. = 26


## 3. Биннеры — fit / transform (sklearn-стиль)
`NumBinner` (числовой) и `CatBinner` (категориальный) — самостоятельные классы. Учим бины на базе, применяем ко всем данным. Бининг ничего не знает про PSI — годится и для WoE.

In [71]:
# числовой: спайк нулей -> точечный бин, интервалы не пересекают точку
NumBinner(n_bins=10).fit_transform(df['f__txn_count']).value_counts().sort_index()

f__txn_count
(-inf, 0.0)       0
0.0            6473
(0.0, 2.0]     1152
(2.0, 3.0)        0
3.0            1240
(3.0, 4.0]     1152
(4.0, 5.0]      938
(5.0, 6.0]      577
(6.0, 7.0]      286
(7.0, inf]      182
missing           0
Name: count, dtype: int64

In [72]:
# f__rate: спайк 0.5 В СЕРЕДИНЕ; f__num_products: мало уникальных -> дискретный режим
print('rate :', list(NumBinner().fit_transform(df['f__rate']).cat.categories))
print('num_products :', list(NumBinner().fit_transform(df['f__num_products']).cat.categories))

rate : [Interval(-inf, 0.224, closed='right'), Interval(0.224, 0.307, closed='right'), Interval(0.307, 0.374, closed='right'), Interval(0.374, 0.439, closed='right'), Interval(0.439, 0.499, closed='right'), Interval(0.499, 0.5, closed='neither'), 0.5, Interval(0.5, 0.557, closed='right'), Interval(0.557, 0.621, closed='right'), Interval(0.621, 0.688, closed='right'), Interval(0.688, 0.774, closed='right'), Interval(0.774, inf, closed='right'), 'missing']
num_products : [1.0, 2.0, 3.0, 4.0, 5.0, 'other', 'missing']


In [73]:
# тип фичи выбираем ЯВНО (не по dtype): число-кодированная категория
codes = pd.Series(np.random.default_rng(0).integers(1000, 1030, size=len(df)))
print('NumBinner ->', len(NumBinner().fit_transform(codes).cat.categories),
      'бинов (квантили — неверно)')
print('CatBinner ->', len(CatBinner().fit_transform(codes).cat.categories),
      'бинов (категории — верно)')

NumBinner -> 11 бинов (квантили — неверно)
CatBinner -> 32 бинов (категории — верно)


## 4. Выбор базы (reference)
`psi.define_base_data` (переехала из удалённого reference.py): `base_size` (доля), `shift_size` (сдвиг окна), `mask_col` (готовая разметка). `None` -> вся выборка.

In [74]:
for kw in [{}, {'base_size': 0.2}, {'base_size': 0.2, 'shift_size': 0.2}]:
    m = psi.define_base_data(df, **kw)
    print(f'{str(kw):45s} -> {int(m.sum()):5d} строк')

{}                                            -> 12000 строк
{'base_size': 0.2}                            ->  2400 строк
{'base_size': 0.2, 'shift_size': 0.2}         ->  2400 строк


## 5. Распределения бинов и PSI по периодам
База — первые 20% по дате (`psi_base_size=0.2`). Тип фичи — через `is_category`. Сверху доли бинов (линии), снизу PSI с порогами `0.10` / `0.25`.

In [76]:
CAT = set(CATEGORICAL)
for feat in FEATURES:
    psi_plots.plot_feature(df, feat, is_category=(feat in CAT), psi_base_size=0.2).show()

## 6. Сводная таблица PSI по периодам

In [ ]:
psi_tbl = psi.calc_psi_by_features(df, FEATURES, cat_features=CATEGORICAL, psi_base_size=0.2)

def _hl(v):
    if v >= psi_plots.PSI_ALERT: return 'background-color:#f8b4b4'  # >=0.25 alert
    if v >= psi_plots.PSI_WARN:  return 'background-color:#fde68a'  # >=0.10 warn
    return ''

psi_tbl.round(4).style.map(_hl)

## 7. Архитектура

- **`binning/`** — `NumBinner` / `CatBinner` (fit/transform), тип выбирается ЯВНО, независим от PSI → переиспользуем для WoE: `NumBinner().fit(train); .transform(any)`.
- **`psi.py`** — `define_base_data` (выбор базы: `base_size` / `shift_size` / `mask_col`) + `calc_psi_by_features` / `calc_psi_by_period`; параметры с префиксами `binner_*` / `psi_base_*`.
- **Краевые случаи:** спайки супервстречаемых значений (`= v`, интервалы не пересекают точку), дискретный режим (мало уникальных), слияние редких бинов (`min_bin="auto"`), Laplace-сглаживание.